# Speculative Decoding Experiment (Colab / T4)

이 노트북은 **draft 모델**이 제안한 토큰들을 **target 모델**이 검증(accept/reject)하는 **speculative decoding**을 구현하고, `k`(한 라운드에 제안할 토큰 수)에 따른 효율을 간단히 실험합니다.

**기본 실험 모델**
- Target: `gpt2`
- Draft: `sshleifer/tiny-gpt2`

GPU 런타임(T4)에서 실행하세요: `런타임 → 런타임 유형 변경 → GPU`.


In [1]:
#@title 0) GPU 확인
!nvidia-smi -L
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


GPU 0: Tesla T4 (UUID: GPU-d0976aa4-1a0e-a076-1e44-bc98e666346e)
torch: 2.9.0+cu128
cuda available: True
gpu: Tesla T4


In [2]:
#@title 1) 의존성 설치 (안정 버전)
!pip -q install -U "transformers==4.41.2" "tokenizers==0.19.1" "accelerate>=0.30" "sentencepiece"
print('done')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
done


In [3]:
#@title 2) 모델 로드
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TARGET_MODEL = "gpt2-medium"
DRAFT_MODEL  = "gpt2"

tok = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

target = AutoModelForCausalLM.from_pretrained(TARGET_MODEL).to(device)
draft  = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL).to(device)
target.eval(); draft.eval()

print('loaded:', TARGET_MODEL, 'and', DRAFT_MODEL)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

loaded: gpt2-medium and gpt2


## 3) Speculative Decoding 구현

간단한 greedy 기반 구현입니다.
- Draft가 한 번에 `k`개 토큰을 제안
- Target이 같은 구간을 한 번에 forward해서 토큰을 비교
- 일치하는 prefix만 accept
- reject가 발생하면 해당 위치에서 target 토큰 1개를 채택하고 라운드 종료

⚠️ 연구/논문 수준의 완전한 구현(확률적 accept-reject, sampling 등)보다 **실험/학습 목적의 안정적인 baseline**에 초점을 맞췄습니다.


In [4]:
#@title 3) Speculative decoding (greedy verify)
import time
from dataclasses import dataclass

@dataclass
class SpecDecodeStats:
    generated: int
    accepted: int
    rounds: int
    time_sec: float

def greedy_next_token(model, input_ids, past_key_values=None):
    """Return next token (greedy) and updated cache."""
    with torch.no_grad():
        out = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
        logits = out.logits[:, -1, :]
        nxt = torch.argmax(logits, dim=-1, keepdim=True)
        return nxt, out.past_key_values

def draft_propose_k(draft_model, prefix_ids, k, past=None):
    """Propose k tokens from draft (greedy), returning proposed tokens and final cache."""
    proposed = []
    cur_ids = prefix_ids
    cur_past = past
    for _ in range(k):
        nxt, cur_past = greedy_next_token(draft_model, cur_ids[:, -1:], past_key_values=cur_past) if cur_past is not None else greedy_next_token(draft_model, cur_ids)
        proposed.append(nxt)
        cur_ids = torch.cat([cur_ids, nxt], dim=1)
    return torch.cat(proposed, dim=1), cur_past

def target_verify(target_model, prefix_ids, proposed_tokens, past=None):
    """Run target on prefix+proposed to get greedy tokens for each step and cache."""
    with torch.no_grad():
        # If we have past, only feed the newly proposed segment token-by-token in one shot is tricky.
        # Simplicity: feed full sequence (prefix + proposed). For small prompts/lengths this is OK for Colab experiments.
        full = torch.cat([prefix_ids, proposed_tokens], dim=1)
        out = target_model(input_ids=full, use_cache=True)
        # logits for positions that predict each proposed token are the logits at the end of each step.
        # For token t_i in proposed, the model predicts it at position prefix_len+i-1.
        logits = out.logits  # [B, T, V]
        prefix_len = prefix_ids.shape[1]
        # predicted tokens corresponding to each proposed position
        preds = torch.argmax(logits[:, prefix_len-1:prefix_len-1+proposed_tokens.shape[1], :], dim=-1)
        return preds, out.past_key_values

def speculative_decode_greedy(prompt, max_new_tokens=50, k=4, seed=0):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    input_ids = tok(prompt, return_tensors='pt').input_ids.to(device)
    accepted = 0
    rounds = 0

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()

    while input_ids.shape[1] < tok(prompt, return_tensors='pt').input_ids.shape[1] + max_new_tokens:
        rounds += 1
        # 1) draft proposes k tokens
        proposed, _ = draft_propose_k(draft, input_ids, k, past=None)
        # 2) target verifies
        preds, _ = target_verify(target, input_ids, proposed, past=None)
        # 3) accept longest prefix
        match = (preds == proposed)
        # count consecutive True from start
        m = 0
        for j in range(match.shape[1]):
            if bool(match[0, j].item()):
                m += 1
            else:
                break
        if m > 0:
            input_ids = torch.cat([input_ids, proposed[:, :m]], dim=1)
            accepted += m
        # if reject happened before consuming all k, take target token at first mismatch
        if m < proposed.shape[1]:
            # the target's predicted token at that position
            reject_tok = preds[:, m:m+1]
            input_ids = torch.cat([input_ids, reject_tok], dim=1)
        # stop if eos
        if int(input_ids[0, -1].item()) == tok.eos_token_id:
            break

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.time()
    gen = input_ids.shape[1] - tok(prompt, return_tensors='pt').input_ids.shape[1]
    return tok.decode(input_ids[0], skip_special_tokens=True), SpecDecodeStats(
        generated=int(gen), accepted=int(accepted), rounds=int(rounds), time_sec=float(t1-t0)
    )


In [5]:
#@title 4) 단일 실행 테스트
prompt = "The future of AI is"
text, stats = speculative_decode_greedy(prompt, max_new_tokens=40, k=4, seed=0)
print(text)
print(stats)


The future of AI is not just about how we use it, but how we use it to improve our lives.

The future of AI is not just about how we use it, but how we use it to improve our lives.
SpecDecodeStats(generated=43, accepted=37, rounds=14, time_sec=6.065547704696655)


In [6]:
#@title 5) k sweep 실험 (간단 리포트)
import pandas as pd

prompt = "The future of AI is"
max_new = 40
seeds = list(range(10))  # 필요하면 50으로 늘리기
ks = [1,2,4,8]

rows = []
for k in ks:
    acc_rates = []
    tok_per_sec = []
    rounds_list = []
    for s in seeds:
        _, st = speculative_decode_greedy(prompt, max_new_tokens=max_new, k=k, seed=s)
        acc_rates.append(st.accepted / max(st.generated, 1))
        tok_per_sec.append(st.generated / max(st.time_sec, 1e-9))
        rounds_list.append(st.rounds)
    rows.append({
        'k': k,
        'accept_rate_mean': float(sum(acc_rates)/len(acc_rates)),
        'tok_per_sec_mean': float(sum(tok_per_sec)/len(tok_per_sec)),
        'rounds_mean': float(sum(rounds_list)/len(rounds_list)),
    })

df = pd.DataFrame(rows)
df


,k,accept_rate_mean,tok_per_sec_mean,rounds_mean
0,1,0.850000,22.695929,40.0
1,2,0.853659,32.519975,22.0
2,4,0.860465,46.937994,14.0
3,8,0.872340,43.714597,10.0
